In [1]:
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'pandas'

# **Dataset Cleaning Pipeline**

A data-driven analysis of the German residential property market combining city-level transaction data with national macro-economic indicators. The pipeline produces two master datasets:

1. **City-Level Master Dataset** - Quarterly property price data across 21 German cities, 2005-2024
2. **Macro-Economic Master Dataset** - Annual national indicators covering labour market, inflation, housing supply, credit conditions and affordability stress, 2005-2025

Both datasets are designed to be used together. The recommended analytical window where all variables are complete across both datasets is **2007-2024**.

# **City-Level Master Dataset**

A unified quarterly dataset covering residential property transaction prices across 21 German cities from 2005 to 2024. Built by integrating three sources (GREIX Nominal Dataset, City Metrics Dataset, GREIX Affordability Indicators). No comparable unified dataset exists on Kaggle or public platforms that combines GREIX transaction prices with city-level distribution metrics and affordability ratios in a single clean file. 

**Cities** - 21 German Cities 
**Property Types** - single_family, apartment, multi_family 
**Time Range** - Q1 2005 - Q4 2024
**Frequency** - Quarterly
**Total Rows** - ~30,000

**Column Dictionary (| Column | Description | Unit | Source |)**

- | city | City name (standardised) | - | GREIX |
- | date | Quarter start data | YYYY-MM-DD | GREIX |
- | year | Calendar year | - | Derived |
- | quarter | Quarter number (1-4) | - | Derived |
- | property_type | single_family / apartment / multi_family | - | GREIX | 
- | price_m2 | Transaction per square metre | EUR/msq. | GREIX |
- | avg_price_m2 | Average price per square metre within city-quarter | EUR/msq. | City Metrics |
- | med_price_m2 | Median price per square metre within city-quarter | EUR/msq. | City Metrics |
- | p75_price_m2 | 75th percentile price per square metre | EUR/msq. | City Metrics |
- | p25_price_m2 | 25th percentile price per square metre | EUR/msq. | City Metrics |
- | affordability_ratio | Mortgage cost as share of income (MCR) | Ratio | GREIX Affordability |

**Known NaN Values (| Column | NaN Count | Reason |)**

- | avg_price_m2 | ~3,288 | City Metrics source only available from 2012 onwards |
- | med_price_m2 | ~3,288 | City Metrics source only available from 2012 onwards |
- | p75_price_m2 | ~3,288 | City Metrics source only available from 2012 onwards |
- | p25_price_m2 | ~3,288 | City Metrics source only available from 2012 onwards | 
- | affordability_ratio | ~904 | Annual metric joined to quarterly data, some city/year combination not covered in source |

**Methodoloy**

- Affordability ratio reflects MCR (Hypothekenquote) for apartments (ETW) only. Applied uniformly across all property types as no segment-specfic ratio was available for cities
- Affordabilit ratio is annual, all quarters within the same city/year share the same value. This is expected behaviour from joining annual data to quarterly data, not a data error.
- City name standardisation for Frankfurt->Frankfurt am Main, Cologne->Köln, Luebeck->Lübeck, Munich->München, Muenster->Münster, Duesseldorf->Düsseldorf
- North Rhine-Westphalia state-level row removed

# GREIX Nominal Dataset

**GREIX Housing Price Dataset** - [(Source)](https://)

- Provides quarterly housing price data for major German cities
- Index constructed using Transaction Data from Expert Appraisal Committees
- Uses Hedonic Regression Methods to control for property characteristics
- Serves as the Primary Housing Market Dataset

In [3]:
# Dataset Transformation for GREIX Nominal Dataset

greix = pd.read_excel("data_raw/GREIX_nominal.xlsx")

# Inspecting data shape

print("GREIX Shape:", greix.shape)

# Inspecting Columns

print("GREIX Columns:", greix.columns.tolist())

# Inspecting Data Types

print("GREIX Data Types:\n", greix.dtypes)

# Inspecting Missing Values

print("GREIX Missing Values:\n", greix.isnull().sum())

# Dataset Preview

print("GREIX Preview:\n", greix.head())


GREIX Shape: (80, 56)
GREIX Columns: ['date', 'Berlin (Single-family house)', 'Berlin (Apartment)', 'Berlin (Multi-family house)', 'Bonn (Single-family house)', 'Bonn (Apartment)', 'Bochum (Single-family house)', 'Bochum (Apartment)', 'Bochum (Multi-family house)', 'Chemnitz (Apartment)', 'Cologne (Single-family house)', 'Cologne (Apartment)', 'Cologne (Multi-family house)', 'Duesseldorf (Single-family house)', 'Duesseldorf (Apartment)', 'Duesseldorf (Multi-family house)', 'Dresden (Single-family house)', 'Dresden (Apartment)', 'Dortmund (Single-family house)', 'Dortmund (Apartment)', 'Duisburg (Single-family house)', 'Duisburg (Apartment)', 'Duisburg (Multi-family house)', 'Erfurt (Single-family house)', 'Erfurt (Apartment)', 'Frankfurt (Single-family house)', 'Frankfurt (Apartment)', 'Frankfurt (Multi-family house)', 'GREIX (Single-family house)', 'GREIX (Apartment)', 'GREIX (Multi-family house)', 'Hamburg (Single-family house)', 'Hamburg (Apartment)', 'Hamburg (Multi-family house)',

In [4]:
# Removing Aggregated Columns 

greix = greix.loc[:, ~greix.columns.str.contains("GREIX")]

print("GREIX Shape after removing aggregated columns:", greix.shape)

GREIX Shape after removing aggregated columns: (80, 53)


In [5]:
# Reshaping from Wide-Format to Long-Format

greix_long = greix.melt(id_vars="date", var_name="city_property", value_name="price_m2")

greix_long.head()

,date,city_property,price_m2
0,Q1 2005,Berlin (Single-family house),NaN
1,Q2 2005,Berlin (Single-family house),NaN
2,Q3 2005,Berlin (Single-family house),NaN
3,Q4 2005,Berlin (Single-family house),NaN
4,Q1 2006,Berlin (Single-family house),NaN


In [6]:
# Splitting City and Property Type

greix_long[["city", "property_type"]] = greix_long["city_property"].str.extract(r"(.+) \((.+)\)")
greix_long = greix_long.drop(columns="city_property")

greix_long.head()

,date,price_m2,city,property_type
0,Q1 2005,NaN,Berlin,Single-family house
1,Q2 2005,NaN,Berlin,Single-family house
2,Q3 2005,NaN,Berlin,Single-family house
3,Q4 2005,NaN,Berlin,Single-family house
4,Q1 2006,NaN,Berlin,Single-family house


In [7]:
# Removing Missing Price rows

greix_long = greix_long.dropna(subset=["price_m2"])

print("GREIX Shape after removing missing price rows:", greix_long.shape)

GREIX Shape after removing missing price rows: (3752, 4)


In [ ]:
# Reformatting String

greix_long["date"] = greix_long["date"].str.replace(r"Q([1-4]) (\d{4})", r"\2Q\1", regex=True)

# Proper Date Encoding

greix_long["date"] = pd.PeriodIndex(greix_long["date"], freq="Q").to_timestamp()


In [13]:
greix_long.head()

,date,price_m2,city,property_type
240,2005-01-01,2000.0,Bonn,Single-family house
241,2005-04-01,1900.0,Bonn,Single-family house
242,2005-07-01,1900.0,Bonn,Single-family house
243,2005-10-01,1800.0,Bonn,Single-family house
244,2006-01-01,1900.0,Bonn,Single-family house


In [14]:
# Adding Derived Columns

greix_long["year"] = greix_long["date"].dt.year
greix_long["quarter"] = greix_long["date"].dt.quarter

# Reordering Columns

greix_clean = greix_long[["city", "date", "year", "quarter", "property_type", "price_m2"]]

greix_clean.head()

,city,date,year,quarter,property_type,price_m2
240,Bonn,2005-01-01,2005,1,Single-family house,2000.0
241,Bonn,2005-04-01,2005,2,Single-family house,1900.0
242,Bonn,2005-07-01,2005,3,Single-family house,1900.0
243,Bonn,2005-10-01,2005,4,Single-family house,1800.0
244,Bonn,2006-01-01,2006,1,Single-family house,1900.0


In [19]:
# Standardizing Property Names

greix_clean["property_type"] = greix_clean["property_type"].replace({"Single-family house": "single_family", "Multi-family house": "multi_family", "Apartment": "apartment"})

greix_clean.head()

,city,date,year,quarter,property_type,price_m2
240,Bonn,2005-01-01,2005,1,single_family,2000.0
241,Bonn,2005-04-01,2005,2,single_family,1900.0
242,Bonn,2005-07-01,2005,3,single_family,1900.0
243,Bonn,2005-10-01,2005,4,single_family,1800.0
244,Bonn,2006-01-01,2006,1,single_family,1900.0


In [20]:
# Saving Cleaned GREIX Dataset

greix_clean.to_csv("data_clean/greix_clean.csv", index=False)

# City Metrics Dataset

**City Housing Metrics Dataset** [(Source)](https://)

- Contains summary statistics describing the Distribution of Housing Prices within each city
- Provides Percentile-based measures
- Helps analyze Price Inequality and Market Dispersion

In [21]:
# Dataset Transformation for City Metrics Dataset

city_metrics = pd.read_excel("data_raw/City_metrics_public.xlsx")

# Inspecting data shape

print("City Metrics Shape:", city_metrics.shape)

# Inspecting Columns

print("City Metrics Columns:", city_metrics.columns.tolist())

# Inspecting Data Types

print("City Metrics Data Types:\n", city_metrics.dtypes)

# Inspecting Missing Values

print("City Metrics Missing Values:\n", city_metrics.isnull().sum())

# Dataset Preview

print("City Metrics Preview:\n", city_metrics.head())

City Metrics Shape: (18088, 10)
City Metrics Columns: ['Year', 'Quarter', 'Month', 'City', 'Index', 'AVG_PRICE_SQM', 'MED_PRICE_SQM', 'P75_PRICE_SQM', 'P25_PRICE_SQM', 'Inflation_adjusted']
City Metrics Data Types:
 Year                    int64
Quarter               float64
Month                 float64
City                      str
Index                 float64
AVG_PRICE_SQM         float64
MED_PRICE_SQM         float64
P75_PRICE_SQM         float64
P25_PRICE_SQM         float64
Inflation_adjusted      int64
dtype: object
City Metrics Missing Values:
 Year                     0
Quarter               1064
Month                 5320
City                     0
Index                    0
AVG_PRICE_SQM            0
MED_PRICE_SQM            0
P75_PRICE_SQM            0
P25_PRICE_SQM            0
Inflation_adjusted       0
dtype: int64
City Metrics Preview:
    Year  Quarter  Month      City      Index  AVG_PRICE_SQM  MED_PRICE_SQM  \
0  2012      1.0    1.0    Aachen  89.227844           7

In [22]:
# Removing Months

city_metrics_clean = city_metrics.drop(columns="Month")

print("City Metrics Shape after removing Month column:", city_metrics_clean.shape)

City Metrics Shape after removing Month column: (18088, 9)


In [23]:
# Remove Inflation Adjusted Rows

city_metrics_clean = city_metrics_clean[city_metrics_clean["Inflation_adjusted"] == 0]
city_metrics_clean = city_metrics_clean.drop(columns="Inflation_adjusted")

print("City Metrics Shape after removing Inflation_adjusted column:", city_metrics_clean.shape)

City Metrics Shape after removing Inflation_adjusted column: (9044, 8)


In [26]:
city_metrics_clean.head()

,Year,Quarter,City,Index,AVG_PRICE_SQM,MED_PRICE_SQM,P75_PRICE_SQM,P25_PRICE_SQM,quarter_str
0,2012,1.0,Aachen,89.227844,7.10,6.86,8.33,5.95,2012Q1.0
2,2012,1.0,Augsburg,84.383804,7.00,7.00,7.62,6.33,2012Q1.0
4,2012,1.0,Berlin,83.126205,7.13,6.76,8.04,5.85,2012Q1.0
6,2012,1.0,Bielefeld,92.340942,5.84,5.71,6.40,5.11,2012Q1.0
8,2012,1.0,Bocholt,90.769547,5.81,5.42,6.58,5.23,2012Q1.0


In [29]:
# Proper Date Encoding

city_metrics_clean = city_metrics_clean.dropna(subset=["Quarter"])
city_metrics_clean["Quarter"] = city_metrics_clean["Quarter"].astype(int)
city_metrics_clean["quarter_str"] = (city_metrics_clean["Year"].astype(str) + "Q" + city_metrics_clean["Quarter"].astype(str))
city_metrics_clean["date"] = pd.PeriodIndex(city_metrics_clean["quarter_str"], freq="Q").to_timestamp()
city_metrics_clean = city_metrics_clean.drop(columns="quarter_str")

city_metrics_clean.head()

,Year,Quarter,City,Index,AVG_PRICE_SQM,MED_PRICE_SQM,P75_PRICE_SQM,P25_PRICE_SQM,date
0,2012,1,Aachen,89.227844,7.10,6.86,8.33,5.95,2012-01-01
2,2012,1,Augsburg,84.383804,7.00,7.00,7.62,6.33,2012-01-01
4,2012,1,Berlin,83.126205,7.13,6.76,8.04,5.85,2012-01-01
6,2012,1,Bielefeld,92.340942,5.84,5.71,6.40,5.11,2012-01-01
8,2012,1,Bocholt,90.769547,5.81,5.42,6.58,5.23,2012-01-01


In [31]:
# Standardizing Column Names

city_metrics_clean = city_metrics_clean.rename(columns={"Year": "year", "Quarter": "quarter", "City": "city", "AVG_PRICE_SQM": "avg_price_sqm", "MED_PRICE_SQM": "med_price_sqm", "P75_PRICE_SQM": "p75_price_sqm", "P25_PRICE_SQM": "p25_price_sqm"})

city_metrics_clean.shape
city_metrics_clean.head()

,year,quarter,city,Index,avg_price_sqm,med_price_sqm,p75_price_sqm,p25_price_sqm,date
0,2012,1,Aachen,89.227844,7.10,6.86,8.33,5.95,2012-01-01
2,2012,1,Augsburg,84.383804,7.00,7.00,7.62,6.33,2012-01-01
4,2012,1,Berlin,83.126205,7.13,6.76,8.04,5.85,2012-01-01
6,2012,1,Bielefeld,92.340942,5.84,5.71,6.40,5.11,2012-01-01
8,2012,1,Bocholt,90.769547,5.81,5.42,6.58,5.23,2012-01-01


In [32]:
# Saving Cleaned City Metrics Dataset

city_metrics_clean.to_csv("data_clean/city_metrics_clean.csv", index=False)

# City Overlap Check (GREIX, City Metrics)

In [33]:
greix_cities = set(greix_clean["city"].unique())
metrics_cities = set(city_metrics_clean["city"].unique())

print("Cities in GREIX dataset:", greix_cities)
print("Cities in City Metrics dataset:", metrics_cities)

print("Cities in both datasets:", greix_cities.intersection(metrics_cities))

print("Cities only in GREIX dataset:", greix_cities.difference(metrics_cities))
print("Cities only in City Metrics dataset:", metrics_cities.difference(greix_cities))

print("Overlap:", len(greix_cities.intersection(metrics_cities)))

Cities in GREIX dataset: {'Dortmund', 'Dresden', 'Stuttgart', 'Leipzig', 'Duisburg', 'Karlsruhe', 'Luebeck', 'Frankfurt', 'North Rhine-Westphalia', 'Hamburg', 'Duesseldorf', 'Bonn', 'Rhein-Erft-Kreis', 'Erfurt', 'Chemnitz', 'Cologne', 'Potsdam', 'Bochum', 'Wiesbaden', 'Kreis Mettmann', 'Munich', 'Muenster'}
Cities in City Metrics dataset: {'Augsburg', 'Braunschweig', 'Düsseldorf', 'Dortmund', 'Hamm', 'Hannover', 'Köln', 'Stuttgart', 'Leipzig', 'Duisburg', 'Karlsruhe', 'Frankfurt am Main', 'Nürnberg', 'Bocholt', 'Gelsenkirchen', 'Mönchengladbach', 'Essen', 'Wuppertal', 'Mannheim', 'Greix', 'Hamburg', 'München', 'Bremen', 'Bielefeld', 'Münster', 'Bonn', 'Rhein-Erft-Kreis', 'Lübeck', 'Erfurt', 'Chemnitz', 'Potsdam', 'Bochum', 'Wiesbaden', 'Kiel', 'Kreis Mettmann', 'Aachen', 'Dresden', 'Berlin'}
Cities in both datasets: {'Bonn', 'Duisburg', 'Rhein-Erft-Kreis', 'Erfurt', 'Chemnitz', 'Potsdam', 'Karlsruhe', 'Bochum', 'Dortmund', 'Wiesbaden', 'Hamburg', 'Kreis Mettmann', 'Stuttgart', 'Dresden

In [34]:
# Removing State-Level Row

greix_clean = greix_clean[greix_clean["city"] != "North Rhine-Westphalia"]

# Standardizing GREIX City Names

city_map = {
    "Frankfurt": "Frankfurt am Main", 
    "Cologne": "Köln",
    "Luebeck": "Lübeck", 
    "Munich": "München", 
    "Muenster": "Münster",
    "Duesseldorf": "Düsseldorf"
}

greix_clean["city"] = greix_clean["city"].replace(city_map)

In [35]:
greix_cities = set(greix_clean["city"].unique())
metrics_cities = set(city_metrics_clean["city"].unique())

print("Overlap:", len(greix_cities.intersection(metrics_cities)))
print("Cities only in GREIX Dataset:", greix_cities.difference(metrics_cities))

Overlap: 21
Cities only in GREIX Dataset: set()


# Merging GREIX and City Metrics to form a Partial City-Level Dataset

In [36]:
# Merge

city_master = greix_clean.merge(city_metrics_clean, on=["city", "date"], how="left")

city_master.head()
city_master.shape

(10616, 13)

In [37]:
city_master.head()

,city,date,year_x,quarter_x,property_type,price_m2,year_y,quarter_y,Index,avg_price_sqm,med_price_sqm,p75_price_sqm,p25_price_sqm
0,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bonn,2005-04-01,2005,2,single_family,1900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bonn,2005-07-01,2005,3,single_family,1900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bonn,2005-10-01,2005,4,single_family,1800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bonn,2006-01-01,2006,1,single_family,1900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# Cleaning the Merged Dataset

city_master = city_master.drop(columns=["year_y", "quarter_y"])

city_master = city_master.rename(columns={
    "year_x": "year",
    "quarter_x": "quarter",
    "avg_price_sqm": "avg_price_m2",
    "med_price_sqm": "med_price_m2",
    "p75_price_sqm": "p75_price_m2",
    "p25_price_sqm": "p25_price_m2"
})

city_master.head()

,city,date,year,quarter,property_type,price_m2,Index,avg_price_m2,med_price_m2,p75_price_m2,p25_price_m2
0,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,NaN
1,Bonn,2005-04-01,2005,2,single_family,1900.0,NaN,NaN,NaN,NaN,NaN
2,Bonn,2005-07-01,2005,3,single_family,1900.0,NaN,NaN,NaN,NaN,NaN
3,Bonn,2005-10-01,2005,4,single_family,1800.0,NaN,NaN,NaN,NaN,NaN
4,Bonn,2006-01-01,2006,1,single_family,1900.0,NaN,NaN,NaN,NaN,NaN


In [39]:
city_master = city_master.drop(columns="Index")

city_master.head()

,city,date,year,quarter,property_type,price_m2,avg_price_m2,med_price_m2,p75_price_m2,p25_price_m2
0,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN
1,Bonn,2005-04-01,2005,2,single_family,1900.0,NaN,NaN,NaN,NaN
2,Bonn,2005-07-01,2005,3,single_family,1900.0,NaN,NaN,NaN,NaN
3,Bonn,2005-10-01,2005,4,single_family,1800.0,NaN,NaN,NaN,NaN
4,Bonn,2006-01-01,2006,1,single_family,1900.0,NaN,NaN,NaN,NaN


In [41]:
# Checking Missing Values

print("Missing Values in Merged Dataset:\n", city_master.isnull().sum())

city_master.groupby("year")[["avg_price_m2"]].count()

Missing Values in Merged Dataset:
 city                0
date                0
year                0
quarter             0
property_type       0
price_m2            0
avg_price_m2     1144
med_price_m2     1144
p75_price_m2     1144
p25_price_m2     1144
dtype: int64


,avg_price_m2
year,
2005,0
2006,0
2007,0
2008,0
2009,0
2010,0
2011,0
2012,688
2013,688


In [43]:
# Saving Partial City-Level Dataset

city_master.to_csv("data_clean/city_master_partial.csv", index=False)

# Affordability Dataset

**GREIX Affordability Indicators Dataset** [(Source)](https://)

- Measures Housing Affordability across German cities
- Affordability = Housing Cost / Housing Income

In [45]:
# Dataset Transformation for Affordability Dataset

affordability = pd.read_excel("data_raw/affordability_indicators_greix.xlsx")

# Inspecting data shape

print("Affordability Shape:", affordability.shape)

# Inspecting Columns

print("Affordability Columns:", affordability.columns.tolist())

# Inspecting Data Types

print("Affordability Data Types:\n", affordability.dtypes)

# Inspecting Missing Values

print("Affordability Missing Values:\n", affordability.isnull().sum())

# Dataset Preview

print("Affordability Preview:\n", affordability.head())

Affordability Shape: (9018, 7)
Affordability Columns: ['Jahr', 'Stadt', 'Metrik', 'Segment', 'Finanzierungsquote', 'Wert', 'ID']
Affordability Data Types:
 Jahr                    int64
Stadt                     str
Metrik                    str
Segment                   str
Finanzierungsquote        str
Wert                  float64
ID                        str
dtype: object
Affordability Missing Values:
 Jahr                  0
Stadt                 0
Metrik                0
Segment               0
Finanzierungsquote    0
Wert                  0
ID                    0
dtype: int64
Affordability Preview:
    Jahr   Stadt                 Metrik                  Segment  \
0  1984  Berlin  Hypothekenquote (MCR)  Einfamilienhäuser (EFH)   
1  1985  Berlin  Hypothekenquote (MCR)  Einfamilienhäuser (EFH)   
2  1986  Berlin  Hypothekenquote (MCR)  Einfamilienhäuser (EFH)   
3  1987  Berlin  Hypothekenquote (MCR)  Einfamilienhäuser (EFH)   
4  1988  Berlin  Hypothekenquote (MCR)  Einfamili

In [47]:
affordability["Metrik"].unique()

<StringArray>
['Hypothekenquote (MCR)', 'Eigenkapitalquote (UCR)']
Length: 2, dtype: str

In [48]:
affordability["Segment"].unique()

<StringArray>
['Einfamilienhäuser (EFH)', 'Eigentumswohnungen (ETW)']
Length: 2, dtype: str

In [49]:
# Filtering the Dataset for MCR and ETW

affordability_clean = affordability[(affordability["Metrik"] == "Hypothekenquote (MCR)") & (affordability["Segment"] == "Eigentumswohnungen (ETW)")]

affordability_clean.head()

,Jahr,Stadt,Metrik,Segment,Finanzierungsquote,Wert,ID
123,1984,Berlin,Hypothekenquote (MCR),Eigentumswohnungen (ETW),80%,0.294972,1984_Berlin_MCR_ETW_80%
124,1985,Berlin,Hypothekenquote (MCR),Eigentumswohnungen (ETW),80%,0.286996,1985_Berlin_MCR_ETW_80%
125,1986,Berlin,Hypothekenquote (MCR),Eigentumswohnungen (ETW),80%,0.274403,1986_Berlin_MCR_ETW_80%
126,1987,Berlin,Hypothekenquote (MCR),Eigentumswohnungen (ETW),80%,0.253331,1987_Berlin_MCR_ETW_80%
127,1988,Berlin,Hypothekenquote (MCR),Eigentumswohnungen (ETW),80%,0.257126,1988_Berlin_MCR_ETW_80%


In [50]:
# Rename Columns

affordability_clean = affordability_clean.rename(columns={
    "Jahr": "year",
    "Stadt": "city",
    "Wert": "affordability_ratio"
})

# Only Relevant Columns

affordability_clean = affordability_clean[["city", "year", "affordability_ratio"]]

# Standardize City Names

affordability_clean["city"] = affordability_clean["city"].replace(city_map)

In [51]:
affordability_clean.head()

,city,year,affordability_ratio
123,Berlin,1984,0.294972
124,Berlin,1985,0.286996
125,Berlin,1986,0.274403
126,Berlin,1987,0.253331
127,Berlin,1988,0.257126


In [52]:
affordability_clean.shape

(2333, 3)

In [53]:
affordability_clean["city"].nunique()

22

In [60]:
# Saving Cleaned Affordability Dataset

affordability_clean.to_csv("data_clean/affordability_clean.csv", index=False)

# City Overlap Check (Affordability, City-Level Master)

In [54]:
affordability_cities = set(affordability_clean["city"].unique())
master_cities = set(city_master["city"].unique())

print("Cities in Affordability Dataset:", affordability_cities)
print("Cities in Master Dataset:", master_cities)

print("Cities in both datasets:", affordability_cities.intersection(master_cities))

print("Cities only in Affordability dataset:", affordability_cities.difference(master_cities))
print("Cities only in Master dataset:", master_cities.difference(affordability_cities))

print("Overlap:", len(affordability_cities.intersection(master_cities)))

Cities in Affordability Dataset: {'Düsseldorf', 'Dortmund', 'Hamm', 'Stuttgart', 'Leipzig', 'Duisburg', 'Karlsruhe', 'Frankfurt am Main', 'Hamburg', 'München', 'Münster', 'Bonn', 'Bundesweiter Durchschnitt (GREIX)', 'Erfurt', 'Chemnitz', 'Lübeck', 'Potsdam', 'Bochum', 'Wiesbaden', 'Köln', 'Dresden', 'Berlin'}
Cities in Master Dataset: {'Düsseldorf', 'Dortmund', 'Stuttgart', 'Leipzig', 'Duisburg', 'Karlsruhe', 'Frankfurt am Main', 'Hamburg', 'München', 'Münster', 'Bonn', 'Rhein-Erft-Kreis', 'Lübeck', 'Erfurt', 'Chemnitz', 'Potsdam', 'Bochum', 'Wiesbaden', 'Kreis Mettmann', 'Köln', 'Dresden'}
Cities in both datasets: {'Düsseldorf', 'Dortmund', 'Stuttgart', 'Leipzig', 'Duisburg', 'Karlsruhe', 'Frankfurt am Main', 'Hamburg', 'München', 'Münster', 'Bonn', 'Lübeck', 'Erfurt', 'Chemnitz', 'Potsdam', 'Bochum', 'Wiesbaden', 'Köln', 'Dresden'}
Cities only in Affordability dataset: {'Bundesweiter Durchschnitt (GREIX)', 'Berlin', 'Hamm'}
Cities only in Master dataset: {'Kreis Mettmann', 'Rhein-Erf

In [56]:
# Removing National Average

affordability_clean = affordability_clean[affordability_clean["city"] != "Bundesweiter Durchschnitt (GREIX)"]

In [57]:
# Merging the Datasets

city_master_final = city_master.merge(affordability_clean, on=["city", "year"], how="left")

city_master_final.head()

,city,date,year,quarter,property_type,price_m2,avg_price_m2,med_price_m2,p75_price_m2,p25_price_m2,affordability_ratio
0,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,0.178156
1,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,0.222695
2,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,0.129580
3,Bonn,2005-04-01,2005,2,single_family,1900.0,NaN,NaN,NaN,NaN,0.178156
4,Bonn,2005-04-01,2005,2,single_family,1900.0,NaN,NaN,NaN,NaN,0.222695


In [58]:
city_master_final.shape

(30040, 11)

In [59]:
city_master_final.isna().sum()

city                      0
date                      0
year                      0
quarter                   0
property_type             0
price_m2                  0
avg_price_m2           3288
med_price_m2           3288
p75_price_m2           3288
p25_price_m2           3288
affordability_ratio     904
dtype: int64

In [62]:
city_master_final["property_type"].unique()

<StringArray>
['single_family', 'apartment', 'multi_family']
Length: 3, dtype: str

In [ ]:
city_master_final.dtypes

city                              str
date                   datetime64[us]
year                            int32
quarter                         int32
property_type                     str
price_m2                      float64
avg_price_m2                  float64
med_price_m2                  float64
p75_price_m2                  float64
p25_price_m2                  float64
affordability_ratio           float64
dtype: object

In [65]:
# Dropping Duplicates

city_master_final = city_master_final.drop_duplicates()

city_master_final.duplicated().sum()

np.int64(0)

In [66]:
city_master_final.describe()

,date,year,quarter,price_m2,avg_price_m2,med_price_m2,p75_price_m2,p25_price_m2,affordability_ratio
count,30019,30019.000000,30019.000000,30019.000000,26731.000000,26731.000000,26731.000000,26731.000000,29115.000000
mean,2017-05-04 09:12:48.113528,2016.965988,2.499850,2907.681802,9.680423,9.455329,10.889211,8.260503,0.252147
min,2005-01-01 00:00:00,2005.000000,1.000000,500.000000,4.760000,4.750000,5.120000,4.310000,0.073724
25%,2014-01-01 00:00:00,2014.000000,2.000000,1900.000000,7.310000,7.170000,8.080000,6.390000,0.184828
50%,2017-10-01 00:00:00,2017.000000,2.000000,2700.000000,9.350000,9.070000,10.440000,8.000000,0.238238
75%,2021-04-01 00:00:00,2021.000000,3.000000,3700.000000,11.570000,11.280000,13.030000,9.830000,0.302790
max,2024-10-01 00:00:00,2024.000000,4.000000,10400.000000,22.580000,22.070000,25.690000,19.150000,0.587830
std,NaN,4.744615,1.118008,1429.866046,2.907150,2.836254,3.410905,2.348921,0.090992


In [67]:
# Time Coverage by the City-Level Master Dataset

city_master_final["date"].min(), city_master_final["date"].max()

(Timestamp('2005-01-01 00:00:00'), Timestamp('2024-10-01 00:00:00'))

In [ ]:
# Saving Cleaned City-Level Master Dataset

city_master_final.to_csv("data_clean/city_master_dataset.csv", index=False)

: 

# **Macro-Economic Master Dataset**

A unified annual dataset of national macroeconomic indicators for Germany from 2005 to 2025, built from 9 official sources across Destatis and Deutsche Bundesbank. Designed to provide the systemic context explaining residential property price movements in the City-Level Dataset. It combines labour market, wage, inflation, housing supply, credit conditions and affordability stress indicators into a single clean file which barely exists in this form on any public platform.

**Geography** - Germany (National)
**Time Range** - 2005-2025
**Frequency** - Annual
**Total Rows** - 21
**Total Columns** - 21

**Column Dictionary (| Column | Description | Unit | Source |)**

- | year | Calendar Year | - | - |
- | unemployment_rate_pct | Registered unemployment rate | % | Bundesbank |
- | persons_in_employment_mill | Employed persons, annual average | Millions | Destatis |
- | compesntation_bn_eur | Total employee compensation, national | Billion EUR | Bundesbank |
- | avg_compensation_per_employee_eur | Derived average annual compensation | EUR | Derived |
- | earnings_index_monthly_with_bonus | Agreed monthly earnings index inclusive of special pay | 2020=100 | Destatis |
- | earnings_index_monthly_without_bonus | Agreed monthly earnings index exclusive of special pay | 2020=100 | Destatis |
- | cpi_headline | Overall consumer price index | 2020=100 | Destatis |
- | cpi_yoy_change_pct | Annual CPI change | % | Destatis |
- | cpi_housing | Housing-specific CPI (CC13-04) | 2020=100 | Destatis |
- | building_permits_total | All building permits issued | Count | Destatis |
- | building_permits_residential | Residential building permits only | Count | Destatis |
- | hpi__overall | National house price index | 2015=100 | Destatis |
- | hpi_new_property | New residential property price index | 2015=100 | Destatis |
- | hpi_existing_property | Existing residential property price index | 2015=100 | Destatis |
- | mortgage_rate_avg_pct | Annual average mortgage rate | % p.a. | Bundesbank |
- | mortgage_rate_min_pct | Lowest monthly mortgage rate that year | % p.a. | Bundesbank |
- | mortgage_rate_max_pct | Highest monthly mortgage rate that year | % p.a. | Bundesbank |
- | wohngeld_total | Total Wohngeld recipient households | Count | Destatis |
- | wohngeld_rent_support | Rent support recipient households | Count | Destatis |
- | wohngeld_mortgage_support | Mortgage support recipient households | Count | Destatis |

**Known NaN Values (| Column | Years | Reason |)**

- | persons_in_employment_mill | 2005-2006 | IO survey data unavailable before 2007 |
- | avg_compensation_per_employee_eur | 2005-2006 | Derived from employment |
- | earnings_index_monthly_with_bonus | 2005-2009 | Destatis index methodology starts from 2010 |
- | earnings_index_monthly_without_bonus | 2005-2009 | Destatis index methodology starts from 2010 |
- | hpi_overll | 2025 | Destatis HPI not yet published for 2025 |
- | hpi_new_property | 2025 | Destatis HPI not yet published for 2025 |
- | hpi_existing_property | 2025 | Destatis HPI not yet published for 2025 |
- | wohngeld_total | 2025 | Destatis Wohngeld data ends at 2024 |
- | wohngeld_rent_support | Destatis Wohngeld data ends at 2024 |
- | wohngeld_mortgage_support | Destatis Wohngeld data ends at 2024 |

**Methodology**

- Mortgage rates aggregated from monthly to annual using mean, min, and max to capture both average conditions and within-year volatility
- avg_compensation_per_employee_eur is a derived proxy calculated as: (compensation_bn_eur / persons_in_employment_mill) * 1000
- Wohngeld 2023 spike captures that total recipients nearly doubled from 630,965 to 1,148,095 due to the Wohngeld-Plus-Gesetz reform effective January 2023, which significantly expanded eligibility criteria. This is a policy change not a data anomaly.
- Earnings index gap 2022-2024 represents the widening spread between with-bonus and without-bonus indices, reflecting the employer use of tax-free Inflationsausgleichprämie (inflation compensation bonus) rather than permanent base salary increase

**Usage with City-Level Master Dataset**

- Join key 'year'
- Recommended analytical window '2007-2024'
- City dataset drives city-level analysis, macro dataset provides national context
- Example: correlate mortgage_rate_avg_pct with city-level price_m2 trends to explain the 2022-2023 market correction


# Unemployment Rate Dataset

**Registered Unemployment Rate: Germany** - [(Source)](https://www.bundesbank.de/dynamic/action/en/statistics/time-series-databases/time-series-databases/759784/759784?listId=www_ssb_lr_alo)

- Registered unemployment pursuant to Section 16 Social Security Code III
- Sourced from Federal Employment Agency via Bundesbank time series
- Annual frequency, covers unified Germany from 1990 onwards
- Used as: Labour market health indicator

In [2]:
# Dataset Transformation for Unemployment Rate

unemployment = pd.read_csv("data_raw/BBDL1.A.DE1.N.UNE.UBA000.A0000.A01.D00.0.R00.L.csv", skiprows=9, header=None, names=["year", "unemployment_rate_pct", "flags"])

# Inspecting data shape

print("Unemployment Shape:", unemployment.shape)

# Inspecting Columns

print("Unemployment Columns:", unemployment.columns.tolist())

# Inspecting Data Types

print("Unemployment Data Types:\n", unemployment.dtypes)

# Inspecting Missing Values

print("Unemployment Missing Values:\n", unemployment.isnull().sum())

# Dataset Preview

print("Unemployment Preview:\n", unemployment.head())

Unemployment Shape: (77, 3)
Unemployment Columns: ['year', 'unemployment_rate_pct', 'flags']
Unemployment Data Types:
 year                         str
unemployment_rate_pct        str
flags                    float64
dtype: object
Unemployment Missing Values:
 year                      0
unemployment_rate_pct     0
flags                    77
dtype: int64
Unemployment Preview:
           year unemployment_rate_pct  flags
0  last update   2026-03-31 10:04:25    NaN
1         1950                  11.0    NaN
2         1951                  10.4    NaN
3         1952                   9.5    NaN
4         1953                   8.4    NaN


In [3]:
# Filtering to numeric years only to remove any trailing metadata rows

unemployment = unemployment[pd.to_numeric(unemployment["year"], errors="coerce").notna()].copy()

# Converting types

unemployment["year"] = unemployment["year"].astype(int)
unemployment["unemployment_rate_pct"] = pd.to_numeric(unemployment["unemployment_rate_pct"], errors="coerce")

# Dropping flags column since it's not needed in the master dataset

unemployment = unemployment.drop(columns=["flags"])

# Filtering to out target window of 2005-2025

unemployment = unemployment[unemployment["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", unemployment.shape)
print("Year range:", unemployment["year"].min(), "to", unemployment["year"].max())
print("Missing values:\n", unemployment.isnull().sum())
unemployment.head()

Shape after cleaning: (21, 2)
Year range: 2005 to 2025
Missing values:
 year                     0
unemployment_rate_pct    0
dtype: int64


,year,unemployment_rate_pct
0,2005,9.9
1,2006,9.1
2,2007,7.4
3,2008,6.4
4,2009,6.9


In [4]:
# Saving cleaned unemployment dataset

unemployment.to_csv("data_clean/unemployment_clean.csv", index=False)
print("Saved cleaned unemployment dataset to data_clean/unemployment_clean.csv")

Saved cleaned unemployment dataset to data_clean/unemployment_clean.csv


# Compensation of Employees Dataset

**Total Compensation of Employees: Germany** - [(Source)](https://www.bundesbank.de/dynamic/action/en/statistics/time-series-databases/time-series-databases/745582/745582?tsId=BBNZ1.A.DE.N.G.0032.A&statisticType=BBK_ITS&tsTab=0&dateSelect=2025)

- National accounts measure of total domestic wage bill (billions of EUR)
- Residence concept, covering all employees working in Germany
- Annual frequency, sourced from Bundesbank national accounts series
- Used as input for derived wage proxy (combined with persons in employment)

In [5]:
# Dataset Transformation for Compensation of Employees

compensation = pd.read_csv("data_raw/BBNZ1.A.DE.N.G.0032.A.csv", skiprows=7, header=None, names=["year", "compensation_bn_eur", "flags"])

# Inspecting data shape

print("Compensation Shape:", compensation.shape)

# Inspecting Columns

print("Compensation Columns:", compensation.columns.tolist())

# Inspecting Data Types

print("Compensation Data Types:\n", compensation.dtypes)

# Inspecting Missing Values

print("Compensation Missing Values:\n", compensation.isnull().sum())

# Dataset Preview

print("Compensation Preview:\n", compensation.head())

Compensation Shape: (36, 3)
Compensation Columns: ['year', 'compensation_bn_eur', 'flags']
Compensation Data Types:
 year                       str
compensation_bn_eur        str
flags                  float64
dtype: object
Compensation Missing Values:
 year                    0
compensation_bn_eur     0
flags                  36
dtype: int64
Compensation Preview:
           year  compensation_bn_eur  flags
0  last update  2026-02-25 08:07:48    NaN
1         1991              865.317    NaN
2         1992              938.059    NaN
3         1993              960.667    NaN
4         1994              986.420    NaN


In [6]:
# Filtering to numeric years

compensation = compensation[pd.to_numeric(compensation["year"], errors="coerce").notna()].copy()

# Converting types

compensation["year"] = compensation["year"].astype(int)
compensation["compensation_bn_eur"] = pd.to_numeric(compensation["compensation_bn_eur"], errors="coerce")

# Dropping flags column

compensation = compensation.drop(columns=["flags"])

# Filtering target window of 2005-2025

compensation = compensation[compensation["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", compensation.shape)
print("Year range:", compensation["year"].min(), "to", compensation["year"].max())
print("Missing values:\n", compensation.isnull().sum())
compensation.head()

Shape after cleaning: (21, 2)
Year range: 2005 to 2025
Missing values:
 year                   0
compensation_bn_eur    0
dtype: int64


,year,compensation_bn_eur
0,2005,1166.822
1,2006,1189.871
2,2007,1224.813
3,2008,1273.169
4,2009,1281.143


In [7]:
# Saving cleaned compensation dataset

compensation.to_csv("data_clean/compensation_clean.csv", index=False)
print("Saved cleaned compensation dataset to data_clean/compensation_clean.csv")

Saved cleaned compensation dataset to data_clean/compensation_clean.csv


# Mortgage Interest Rates Dataset

**Effective Interest Rates on Housing Loans to Households: Germany** - [(Source)](https://www.bundesbank.de/dynamic/action/en/statistics/time-series-databases/time-series-databases/745582/745582?tsId=BBIM1.M.DE.B.A2C.A.C.A.2250.EUR.N&dateSelect=2026)

- Annual percentage rate of charge on new housing loan agreements
- Covers secured and unsecured loans for home purchase including building loans
- Monthly data aggregated to annual mean, min, and max
- Used as credit conditions and financing cost indicator

In [8]:
# Dataset Transformation for Mortgage Interest Rates

mortgage = pd.read_csv("data_raw/BBIM1.M.DE.B.A2C.A.C.A.2250.EUR.N.csv", skiprows=8, header=None, names=["month", "mortgage_rate_pct", "flags"])

# Inspecting Data Shape

print("Mortgage Shape:", mortgage.shape)

# Inspecting Columns

print("Mortgage Columns:", mortgage.columns.tolist())

# Inspecting Data Types

print("Mortgage Data Types:\n", mortgage.dtypes)

# Inspecting Missing Values

print("Mortgage Missing Values:\n", mortgage.isnull().sum())

# Dataset Preview

print("Mortgage Preview:\n", mortgage.head())

Mortgage Shape: (280, 3)
Mortgage Columns: ['month', 'mortgage_rate_pct', 'flags']
Mortgage Data Types:
 month                str
mortgage_rate_pct    str
flags                str
dtype: object
Mortgage Missing Values:
 month                  1
mortgage_rate_pct      0
flags                278
dtype: int64
Mortgage Preview:
          month    mortgage_rate_pct flags
0  last update  2026-04-01 11:46:50   NaN
1      2003-01                 5.39   NaN
2      2003-02                 5.18   NaN
3      2003-03                 5.07   NaN
4      2003-04                 5.04   NaN


In [9]:
# Filtering to valid monthly rows only

mortgage = mortgage[mortgage["month"].str.match(r"^\d{4}-\d{2}$", na=False)].copy()

# Converting types

mortgage["mortgage_rate_pct"] = pd.to_numeric(mortgage["mortgage_rate_pct"], errors="coerce")

# Extracting year from month string

mortgage["year"] = mortgage["month"].str[:4].astype(int)

# Dropping flags and month columns

mortgage = mortgage.drop(columns=["flags", "month"])

# Filtering to target window of 2005-2025

mortgage = mortgage[mortgage["year"].between(2005, 2025)]

print("Shape after filtering:", mortgage.shape)
print("Nulls:\n", mortgage.isnull().sum())
mortgage.head()

Shape after filtering: (252, 2)
Nulls:
 mortgage_rate_pct    0
year                 0
dtype: int64


,mortgage_rate_pct,year
25,4.55,2005
26,4.49,2005
27,4.49,2005
28,4.50,2005
29,4.40,2005


In [10]:
# Aggregating monthly to annual

mortgage_annual = mortgage.groupby("year").agg(
    mortgage_rate_avg_pct = ("mortgage_rate_pct", "mean"),
    mortgage_rate_min_pct = ("mortgage_rate_pct", "min"),
    mortgage_rate_max_pct = ("mortgage_rate_pct", "max")
).reset_index()

# Rounding to 2 decimal places

mortgage_annual[["mortgage_rate_avg_pct", "mortgage_rate_min_pct", "mortgage_rate_max_pct"]] = mortgage_annual[["mortgage_rate_avg_pct", "mortgage_rate_min_pct", "mortgage_rate_max_pct"]].round(2)

print("Shape after aggregation:", mortgage_annual.shape)
print("Year range:", mortgage_annual["year"].min(), "to", mortgage_annual["year"].max())
print("Missing values:\n", mortgage_annual.isnull().sum())
mortgage_annual.head()

Shape after aggregation: (21, 4)
Year range: 2005 to 2025
Missing values:
 year                     0
mortgage_rate_avg_pct    0
mortgage_rate_min_pct    0
mortgage_rate_max_pct    0
dtype: int64


,year,mortgage_rate_avg_pct,mortgage_rate_min_pct,mortgage_rate_max_pct
0,2005,4.34,4.18,4.55
1,2006,4.69,4.40,4.87
2,2007,5.15,4.85,5.37
3,2008,5.27,4.96,5.54
4,2009,4.34,4.13,4.83


In [11]:
# Saving cleaned mortgage dataset

mortgage_annual.to_csv("data_clean/mortgage_clean.csv", index=False)
print("Saved cleaned mortgage dataset to data_clean/mortgage_clean.csv")

Saved cleaned mortgage dataset to data_clean/mortgage_clean.csv


# Persons in Employment Dataset

**Persons in Employment: Germany (ILO Concept)** - [(Source)](https://www-genesis.destatis.de/datenbank/online/statistic/13231/table/13231-0001/search/s/JTIydW5lbXBsb3llZCUyMg%3D%3D#filter=JTdCJTIyaGlkZUVtcHR5Q29scyUyMiUzQWZhbHNlJTJDJTIyaGlkZUVtcHR5Um93cyUyMiUzQWZhbHNlJTJDJTIyY2FwdGlvbiUyMiUzQSU1QiU3QiUyMnZhcmlhYmxlSWQlMjIlM0ElMjIxMzIzMSUyMiUyQyUyMmlkJTIyJTNBJTIyZmlsdGVyLjAlMjIlMkMlMjJ2YWx1ZXNJZHMlMjIlM0ElNUIlMjIxMzIzMSUyMiU1RCUyQyUyMmNoaWxkcmVuJTIyJTNBJTVCJTdCJTIydmFyaWFibGVJZCUyMiUzQSUyMkRJTlNHJTIyJTJDJTIyaWQlMjIlM0ElMjJmaWx0ZXIuMC4wJTIyJTJDJTIydmFsdWVzSWRzJTIyJTNBJTVCJTIyREclMjIlNUQlMkMlMjJjaGlsZHJlbiUyMiUzQSU1QiU1RCUyQyUyMnNob3dBc0ludGVybGluZSUyMiUzQWZhbHNlJTJDJTIyc2hvd1ZhcmlhYmxlJTIyJTNBZmFsc2UlMkMlMjJzaG93VmFyaWFibGVWYWx1ZSUyMiUzQSU1QiUyMkxBQkVMJTIyJTVEJTJDJTIyc29ydCUyMiUzQSUyMk5hbWVBc2MlMjIlMkMlMjJpc0hpZGRlbiUyMiUzQWZhbHNlJTJDJTIyYmxvY2tDb2RlJTIyJTNBJTIydjElMjIlMkMlMjJwb3NzaWJsZVBsYWNlcyUyMiUzQSU1QiU1RCU3RCU1RCUyQyUyMnNob3dBc0ludGVybGluZSUyMiUzQWZhbHNlJTJDJTIyaXNIaWRkZW4lMjIlM0FmYWxzZSUyQyUyMmJsb2NrQ29kZSUyMiUzQSUyMnMxJTIyJTJDJTIycG9zc2libGVQbGFjZXMlMjIlM0ElNUIlNUQlN0QlNUQlMkMlMjJyb3dIZWFkZXIlMjIlM0ElNUIlN0IlMjJ2YXJpYWJsZUlkJTIyJTNBJTIySkFIUiUyMiUyQyUyMmlkJTIyJTNBJTIycm93VGl0bGUuMCUyMiUyQyUyMnZhbHVlc0lkcyUyMiUzQSU1QiUyMjIwMjUlMjIlMkMlMjIyMDI2JTIyJTJDJTIyMjAyNCUyMiUyQyUyMjIwMjMlMjIlMkMlMjIyMDIyJTIyJTJDJTIyMjAyMSUyMiUyQyUyMjIwMjAlMjIlMkMlMjIyMDE5JTIyJTJDJTIyMjAxOCUyMiUyQyUyMjIwMTclMjIlMkMlMjIyMDE2JTIyJTJDJTIyMjAxNSUyMiUyQyUyMjIwMTQlMjIlMkMlMjIyMDEzJTIyJTJDJTIyMjAxMiUyMiUyQyUyMjIwMTElMjIlMkMlMjIyMDEwJTIyJTJDJTIyMjAwOSUyMiUyQyUyMjIwMDglMjIlMkMlMjIyMDA3JTIyJTJDJTIyMjAwNiUyMiUyQyUyMjIwMDUlMjIlMkMlMjIyMDA0JTIyJTJDJTIyMjAwMyUyMiUyQyUyMjIwMDIlMjIlMkMlMjIyMDAxJTIyJTJDJTIyMjAwMCUyMiU1RCUyQyUyMmNoaWxkcmVuJTIyJTNBJTVCJTdCJTIydmFyaWFibGVJZCUyMiUzQSUyMk1PTkFUJTIyJTJDJTIyaWQlMjIlM0ElMjJyb3dUaXRsZS4wLjAlMjIlMkMlMjJ2YWx1ZXNJZHMlMjIlM0ElNUIlMjJNT05BVDAxJTIyJTJDJTIyTU9OQVQwMiUyMiUyQyUyMk1PTkFUMDMlMjIlMkMlMjJNT05BVDA0JTIyJTJDJTIyTU9OQVQwNSUyMiUyQyUyMk1PTkFUMDYlMjIlMkMlMjJNT05BVDA3JTIyJTJDJTIyTU9OQVQwOCUyMiUyQyUyMk1PTkFUMDklMjIlMkMlMjJNT05BVDEwJTIyJTJDJTIyTU9OQVQxMSUyMiUyQyUyMk1PTkFUMTIlMjIlNUQlMkMlMjJjaGlsZHJlbiUyMiUzQSU1QiU1RCUyQyUyMnNob3dBc0ludGVybGluZSUyMiUzQWZhbHNlJTJDJTIyc2hvd1ZhcmlhYmxlJTIyJTNBdHJ1ZSUyQyUyMnNob3dWYXJpYWJsZVZhbHVlJTIyJTNBJTVCJTIyTEFCRUwlMjIlNUQlMkMlMjJzb3J0JTIyJTNBJTIyQ29kZUFzYyUyMiUyQyUyMmlzSGlkZGVuJTIyJTNBZmFsc2UlMkMlMjJibG9ja0NvZGUlMjIlM0ElMjJ2NCUyMiUyQyUyMnBvc3NpYmxlUGxhY2VzJTIyJTNBJTVCJTdCJTIycHJldlBhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYzJTIyJTJDJTIyaWQlMjIlM0ElMjJyb3dUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEFib3ZlJTIyJTNBbnVsbCUyQyUyMmVsZW1lbnRCZWxvdyUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYzJTIyJTJDJTIyaWQlMjIlM0ElMjJyb3dUaXRsZS4wJTIyJTdEJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBMCUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0FmYWxzZSU3RCUyQyU3QiUyMnByZXZQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MyUyMiUyQyUyMmlkJTIyJTNBJTIycm93VGl0bGUuMCUyMiU3RCUyQyUyMmVsZW1lbnRBYm92ZSUyMiUzQW51bGwlMkMlMjJlbGVtZW50QmVsb3clMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTAlMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBdHJ1ZSU3RCUyQyU3QiUyMnByZXZQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MyUyMiUyQyUyMmlkJTIyJTNBJTIycm93VGl0bGUuMCUyMiU3RCUyQyUyMm5ld1BhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEFib3ZlJTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJlbGVtZW50QmVsb3clMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjMSUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4wJTIyJTdEJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBMCUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0F0cnVlJTdEJTVEJTdEJTVEJTJDJTIyc2hvd0FzSW50ZXJsaW5lJTIyJTNBZmFsc2UlMkMlMjJzaG93VmFyaWFibGUlMjIlM0F0cnVlJTJDJTIyc2hvd1ZhcmlhYmxlVmFsdWUlMjIlM0ElNUIlMjJMQUJFTCUyMiU1RCUyQyUyMmlzSGlkZGVuJTIyJTNBZmFsc2UlMkMlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MyUyMiUyQyUyMnBvc3NpYmxlUGxhY2VzJTIyJTNBJTVCJTdCJTIybmV3UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjQlMjIlMkMlMjJpZCUyMiUzQSUyMnJvd1RpdGxlLjAuMCUyMiU3RCUyQyUyMmVsZW1lbnRBYm92ZSUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnY0JTIyJTJDJTIyaWQlMjIlM0ElMjJyb3dUaXRsZS4wLjAlMjIlN0QlMkMlMjJlbGVtZW50QmVsb3clMjIlM0FudWxsJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBMCUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0FmYWxzZSU3RCUyQyU3QiUyMmVsZW1lbnRBYm92ZSUyMiUzQW51bGwlMkMlMjJlbGVtZW50QmVsb3clMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTAlMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBdHJ1ZSU3RCUyQyU3QiUyMm5ld1BhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEFib3ZlJTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJlbGVtZW50QmVsb3clMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjMSUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4wJTIyJTdEJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBMCUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0F0cnVlJTdEJTVEJTdEJTVEJTJDJTIyY29sdW1uSGVhZGVyJTIyJTNBJTVCJTdCJTIydmFyaWFibGVJZCUyMiUzQSUyMldFUlQwNiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiUyQyUyMnZhbHVlc0lkcyUyMiUzQSU1QiUyMldFUlRPUkclMjIlMkMlMjJYMTNKRFRCJTIyJTJDJTIyQlY0VEIlMjIlNUQlMkMlMjJjaGlsZHJlbiUyMiUzQSU1QiU3QiUyMnZhcmlhYmxlSWQlMjIlM0ElMjJFUlcwMDElMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMCUyMiUyQyUyMnZhbHVlc0lkcyUyMiUzQSU1QiUyMlFNVSUyMiU1RCUyQyUyMmNoaWxkcmVuJTIyJTNBJTVCJTVEJTJDJTIyc2hvd0FzSW50ZXJsaW5lJTIyJTNBZmFsc2UlMkMlMjJpc0hpZGRlbiUyMiUzQWZhbHNlJTJDJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzElMjIlMkMlMjJwb3NzaWJsZVBsYWNlcyUyMiUzQSU1QiU3QiUyMnByZXZQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMm5ld1BhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEFib3ZlJTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMSUyMiU3RCUyQyUyMmVsZW1lbnRCZWxvdyUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMmMzJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wLjIlMjIlN0QlMkMlMjJuZXdTaWJsaW5nSW5kZXglMjIlM0EyJTJDJTIyaGFzVHJhbnNwb3NlUGFydCUyMiUzQWZhbHNlJTdEJTJDJTdCJTIycHJldlBhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIybmV3UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJlbGVtZW50QWJvdmUlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjMyUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4yJTIyJTdEJTJDJTIyZWxlbWVudEJlbG93JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzQlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMyUyMiU3RCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTMlMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBZmFsc2UlN0QlMkMlN0IlMjJwcmV2UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJuZXdQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMmVsZW1lbnRBYm92ZSUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMmM0JTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wLjMlMjIlN0QlMkMlMjJlbGVtZW50QmVsb3clMjIlM0FudWxsJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBNCUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0FmYWxzZSU3RCU1RCU3RCUyQyU3QiUyMnZhcmlhYmxlSWQlMjIlM0ElMjJFUlcwMDIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMSUyMiUyQyUyMnZhbHVlc0lkcyUyMiUzQSU1QiUyMlFNVSUyMiU1RCUyQyUyMmNoaWxkcmVuJTIyJTNBJTVCJTVEJTJDJTIyc2hvd0FzSW50ZXJsaW5lJTIyJTNBZmFsc2UlMkMlMjJpc0hpZGRlbiUyMiUzQWZhbHNlJTJDJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzIlMjIlMkMlMjJwb3NzaWJsZVBsYWNlcyUyMiUzQSU1QiU3QiUyMnByZXZQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMm5ld1BhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEFib3ZlJTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJlbGVtZW50QmVsb3clMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjMSUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4wJTIyJTdEJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBMCUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0FmYWxzZSU3RCUyQyU3QiUyMnByZXZQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMm5ld1BhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEFib3ZlJTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzMlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMiUyMiU3RCUyQyUyMmVsZW1lbnRCZWxvdyUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMmM0JTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wLjMlMjIlN0QlMkMlMjJuZXdTaWJsaW5nSW5kZXglMjIlM0EzJTJDJTIyaGFzVHJhbnNwb3NlUGFydCUyMiUzQWZhbHNlJTdEJTJDJTdCJTIycHJldlBhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIybmV3UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJlbGVtZW50QWJvdmUlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjNCUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4zJTIyJTdEJTJDJTIyZWxlbWVudEJlbG93JTIyJTNBbnVsbCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTQlMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBZmFsc2UlN0QlNUQlN0QlMkMlN0IlMjJ2YXJpYWJsZUlkJTIyJTNBJTIyRVJXMDAzJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wLjIlMjIlMkMlMjJ2YWx1ZXNJZHMlMjIlM0ElNUIlMjJRTVUlMjIlNUQlMkMlMjJjaGlsZHJlbiUyMiUzQSU1QiU1RCUyQyUyMnNob3dBc0ludGVybGluZSUyMiUzQWZhbHNlJTJDJTIyaXNIaWRkZW4lMjIlM0FmYWxzZSUyQyUyMmJsb2NrQ29kZSUyMiUzQSUyMmMzJTIyJTJDJTIycG9zc2libGVQbGFjZXMlMjIlM0ElNUIlN0IlMjJwcmV2UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJuZXdQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMmVsZW1lbnRBYm92ZSUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEJlbG93JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzElMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMCUyMiU3RCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTAlMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBZmFsc2UlN0QlMkMlN0IlMjJwcmV2UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJuZXdQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMmVsZW1lbnRBYm92ZSUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMmMxJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wLjAlMjIlN0QlMkMlMjJlbGVtZW50QmVsb3clMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjMiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4xJTIyJTdEJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBMSUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0FmYWxzZSU3RCUyQyU3QiUyMnByZXZQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMm5ld1BhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEFib3ZlJTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzQlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMyUyMiU3RCUyQyUyMmVsZW1lbnRCZWxvdyUyMiUzQW51bGwlMkMlMjJuZXdTaWJsaW5nSW5kZXglMjIlM0E0JTJDJTIyaGFzVHJhbnNwb3NlUGFydCUyMiUzQWZhbHNlJTdEJTVEJTdEJTJDJTdCJTIydmFyaWFibGVJZCUyMiUzQSUyMkVSVzA4OSUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4zJTIyJTJDJTIydmFsdWVzSWRzJTIyJTNBJTVCJTIyUU1VJTIyJTVEJTJDJTIyY2hpbGRyZW4lMjIlM0ElNUIlNUQlMkMlMjJzaG93QXNJbnRlcmxpbmUlMjIlM0FmYWxzZSUyQyUyMmlzSGlkZGVuJTIyJTNBZmFsc2UlMkMlMjJibG9ja0NvZGUlMjIlM0ElMjJjNCUyMiUyQyUyMnBvc3NpYmxlUGxhY2VzJTIyJTNBJTVCJTdCJTIycHJldlBhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIybmV3UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJlbGVtZW50QWJvdmUlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMmVsZW1lbnRCZWxvdyUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMmMxJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wLjAlMjIlN0QlMkMlMjJuZXdTaWJsaW5nSW5kZXglMjIlM0EwJTJDJTIyaGFzVHJhbnNwb3NlUGFydCUyMiUzQWZhbHNlJTdEJTJDJTdCJTIycHJldlBhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wJTIyJTdEJTJDJTIybmV3UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJlbGVtZW50QWJvdmUlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjMSUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4wJTIyJTdEJTJDJTIyZWxlbWVudEJlbG93JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIyYzIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAuMSUyMiU3RCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTElMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBZmFsc2UlN0QlMkMlN0IlMjJwcmV2UGFyZW50JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjIlMjIlMkMlMjJpZCUyMiUzQSUyMmNvbFRpdGxlLjAlMjIlN0QlMkMlMjJuZXdQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MiUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMCUyMiU3RCUyQyUyMmVsZW1lbnRBYm92ZSUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMmMyJTIyJTJDJTIyaWQlMjIlM0ElMjJjb2xUaXRsZS4wLjElMjIlN0QlMkMlMjJlbGVtZW50QmVsb3clMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJjMyUyMiUyQyUyMmlkJTIyJTNBJTIyY29sVGl0bGUuMC4yJTIyJTdEJTJDJTIybmV3U2libGluZ0luZGV4JTIyJTNBMiUyQyUyMmhhc1RyYW5zcG9zZVBhcnQlMjIlM0FmYWxzZSU3RCU1RCU3RCU1RCUyQyUyMnNob3dBc0ludGVybGluZSUyMiUzQWZhbHNlJTJDJTIyc2hvd1ZhcmlhYmxlJTIyJTNBZmFsc2UlMkMlMjJzaG93VmFyaWFibGVWYWx1ZSUyMiUzQSU1QiUyMkxBQkVMJTIyJTVEJTJDJTIyaXNIaWRkZW4lMjIlM0FmYWxzZSUyQyUyMmJsb2NrQ29kZSUyMiUzQSUyMnYyJTIyJTJDJTIycG9zc2libGVQbGFjZXMlMjIlM0ElNUIlN0IlMjJlbGVtZW50QWJvdmUlMjIlM0FudWxsJTJDJTIyZWxlbWVudEJlbG93JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjMlMjIlMkMlMjJpZCUyMiUzQSUyMnJvd1RpdGxlLjAlMjIlN0QlMkMlMjJuZXdTaWJsaW5nSW5kZXglMjIlM0EwJTJDJTIyaGFzVHJhbnNwb3NlUGFydCUyMiUzQXRydWUlN0QlMkMlN0IlMjJuZXdQYXJlbnQlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2MyUyMiUyQyUyMmlkJTIyJTNBJTIycm93VGl0bGUuMCUyMiU3RCUyQyUyMmVsZW1lbnRBYm92ZSUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnYzJTIyJTJDJTIyaWQlMjIlM0ElMjJyb3dUaXRsZS4wJTIyJTdEJTJDJTIyZWxlbWVudEJlbG93JTIyJTNBJTdCJTIyYmxvY2tDb2RlJTIyJTNBJTIydjQlMjIlMkMlMjJpZCUyMiUzQSUyMnJvd1RpdGxlLjAuMCUyMiU3RCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTAlMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBdHJ1ZSU3RCUyQyU3QiUyMm5ld1BhcmVudCUyMiUzQSU3QiUyMmJsb2NrQ29kZSUyMiUzQSUyMnY0JTIyJTJDJTIyaWQlMjIlM0ElMjJyb3dUaXRsZS4wLjAlMjIlN0QlMkMlMjJlbGVtZW50QWJvdmUlMjIlM0ElN0IlMjJibG9ja0NvZGUlMjIlM0ElMjJ2NCUyMiUyQyUyMmlkJTIyJTNBJTIycm93VGl0bGUuMC4wJTIyJTdEJTJDJTIyZWxlbWVudEJlbG93JTIyJTNBbnVsbCUyQyUyMm5ld1NpYmxpbmdJbmRleCUyMiUzQTAlMkMlMjJoYXNUcmFuc3Bvc2VQYXJ0JTIyJTNBdHJ1ZSU3RCU1RCU3RCU1RCUyQyUyMmZpeEZpcnN0Q29sdW1ucyUyMiUzQWZhbHNlJTdE&sortByValue=default)

- Monthly labour force survey data based on ILO methodology
- Includes all employed persons regardless of hours worked
- Monthly data aggregated to annual mean
- Used as denominator in derived avg compensation per employee calculation
- Note: Reliable data available from 2007 onwards

In [12]:
# Dataset Transformation for Persons in Employment 

employment_raw = pd.read_excel("data_raw/13231-0001_en.xlsx", header=None)

# Inspecting Data Shape

print("Employment Shape:", employment_raw.shape)

# Inspecting Columns

print("Employment Columns:", employment_raw.columns.tolist())

# Inspecting Data Types

print("Employment Data Types:\n", employment_raw.dtypes)

# Inspecting Missing Values

print("Employment Missing Values:\n", employment_raw.isnull().sum())

# Dataset Preview

print("Employment Preview:\n", employment_raw.head())

Employment Shape: (338, 26)
Employment Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
Employment Data Types:
 0        str
1        str
2     object
3        str
4     object
5        str
6     object
7        str
8     object
9        str
10    object
11       str
12    object
13       str
14    object
15       str
16    object
17       str
18    object
19       str
20    object
21       str
22    object
23       str
24    object
25       str
dtype: object
Employment Missing Values:
 0     300
1      14
2      11
3     108
4      12
5     108
6      12
7     108
8      12
9     108
10     11
11     24
12     12
13     24
14     12
15     24
16     12
17     24
18     11
19     24
20     12
21     24
22     12
23     24
24     12
25     24
dtype: int64
Employment Preview:
                                                   0    1   \
0  Unemployed persons, persons in employment, eco...  NaN   
1   Unemployment statistics based on 

/Users/aditya/Desktop/Housing Analysis/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [13]:
# Extracting relevant columns

employment = employment_raw.iloc[5:, [0, 1, 4]].copy()
employment.columns = ["year", "month", "persons_in_employment_mill"]

# Forward-fill year column since year only appears in January row

employment["year"] = employment["year"].ffill()

# Drop rows with missing months

employment = employment[employment["month"].notna()].copy()

# Convert to numeric and replace dashes and non-numeric with NaN

employment["year"] = pd.to_numeric(employment["year"], errors="coerce")
employment["persons_in_employment_mill"] = pd.to_numeric(employment["persons_in_employment_mill"], errors="coerce")

# Drop rows with no employment data

employment = employment.dropna(subset=["persons_in_employment_mill"])

print("Shape after cleaning:", employment.shape)
print("Year range:", employment["year"].min(), "to", employment["year"].max())
print("Missing values:\n", employment.isnull().sum())
employment.head()

Shape after cleaning: (230, 3)
Year range: 2007 to 2026
Missing values:
 year                          0
month                         0
persons_in_employment_mill    0
dtype: int64


,year,month,persons_in_employment_mill
90,2007,January,36.54
91,2007,February,36.34
92,2007,March,36.54
93,2007,April,36.99
94,2007,May,37.10


In [14]:
# Aggregating monthly to annual mean

employment_annual = employment.groupby("year").agg(persons_in_employment_mill = ("persons_in_employment_mill", "mean")).reset_index()
employment_annual["year"] = employment_annual["year"].astype(int)

# Rouding to 3 decimal places

employment_annual["persons_in_employment_mill"] = employment_annual["persons_in_employment_mill"].round(3)

# Filtering to target window of 2005-2025

employment_annual = employment_annual[employment_annual["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after aggregation:", employment_annual.shape)
print("Year range:", employment_annual["year"].min(), "to", employment_annual["year"].max())
print("Missing values:\n", employment_annual.isnull().sum())
employment_annual.head()

Shape after aggregation: (19, 2)
Year range: 2007 to 2025
Missing values:
 year                          0
persons_in_employment_mill    0
dtype: int64


,year,persons_in_employment_mill
0,2007,37.153
1,2008,37.637
2,2009,37.887
3,2010,37.412
4,2011,38.178


In [15]:
# Saving cleaned persons in employment dataset

employment_annual.to_csv("data_clean/employment_clean.csv", index=False)
print("Saved cleaned persons in employment dataset to data_clean/employment_clean.csv")

Saved cleaned persons in employment dataset to data_clean/employment_clean.csv


# Building Permits Dataset

**Building Permits in Structural Engineering: Germany** - [(Source)](https://www-genesis.destatis.de/datenbank/online/statistic/31111/table/31111-0001/search/s/YnVpbGRpbmclMjBwZXJtaXQ%3D#filter=...&sortByValue=default)

- Annual count of approved building permits across Germany
- Covers all building construction activity
- Two series extracted: total buildings and residential buildings only
- Used as housing supply indicator

In [16]:
# Dataset Transformation for Building Permits

permits_raw = pd.read_excel("data_raw/31111-0001_en.xlsx", header=None)

# Inspecting Data Shape

print("Permits Shape:", permits_raw.shape)

# Inspecting Columns

print("Permits Columns:", permits_raw.columns.tolist())

# Inspecting Data Types

print("Permits Data Types:\n", permits_raw.dtypes)

# Inspecting Missing Values

print("Permits Missing Values:\n", permits_raw.isnull().sum())

# Dataset Preview

print("Permits Preview:\n", permits_raw.head())

Permits Shape: (746, 12)
Permits Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Permits Data Types:
 0         str
1      object
2         str
3         str
4     float64
5         str
6     float64
7      object
8         str
9         str
10    float64
11        str
dtype: object
Permits Missing Values:
 0       4
1      42
2     297
3      44
4     746
5      43
6     746
7      44
8     197
9      43
10    746
11     44
dtype: int64
Permits Preview:
                                                   0                        1   \
0  Building permits in structural engineering: Ge...                      NaN   
1                     Statistics of building permits                      NaN   
2                              Building construction                      NaN   
3                                            Germany                      NaN   
4                             Type of building\nYear  Construction activities   

    2    3   4    5   6    7    8    9   10   11  
0 

/Users/aditya/Desktop/Housing Analysis/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [17]:
# Since the file has multiple building type sections stacked vertically, we will need to split the file into separate sections ("Residential and non-residential buildings" - total permits, "Residential buildings" - residential only)

# Extracting all rows where column 0 is a 4-digit year

year_mask = permits_raw[0].astype(str).str.match(r"^\d{4}\.0$|^\d{4}$")
data_rows = permits_raw[year_mask].copy()

# Find section header row indices

total_index = permits_raw[permits_raw[0].astype(str).str.contains("Residential and non-residential buildings", na=False)].index[0]

residential_index = permits_raw[permits_raw[0].astype(str).str.contains("Residential buildings$", na=False)].index[0]

print("Total section starts at row index:", total_index)
print("Residential section starts at row index:", residential_index)

Total section starts at row index: 8
Residential section starts at row index: 34


In [18]:
# Extract total permits section

total_rows = permits_raw.iloc[total_index+1:residential_index].copy()
total_rows = total_rows[pd.to_numeric(total_rows[0], errors="coerce").notna()][[0, 1]].copy()
total_rows.columns = ["year", "building_permits_total"]

# Find next section after residential to bound the slice, Look for non-year header after residential section

next_section_index = permits_raw.iloc[residential_index+1:][permits_raw.iloc[residential_index+1:][0].astype(str).str.match(r"^[A-Z]", na=False)].index[0]

# Extract residential permits section

residential_rows = permits_raw.iloc[residential_index+1:next_section_index].copy()
residential_rows = residential_rows[pd.to_numeric(residential_rows[0], errors="coerce").notna()][[0, 1]].copy()
residential_rows.columns = ["year", "building_permits_residential"]

print("Total permits rows:", len(total_rows))
print("Residential permits rows:", len(residential_rows))
print("\nTotal permits preview:\n", total_rows.head())
print("\nResidential permits preview:\n", residential_rows.head())

Total permits rows: 25
Residential permits rows: 25

Total permits preview:
     year building_permits_total
9   2001                 289794
10  2002                 278340
11  2003                 298787
12  2004                 271944
13  2005                 242123

Residential permits preview:
     year building_permits_residential
35  2001                       234868
36  2002                       230259
37  2003                       254849
38  2004                       227750
39  2005                       198896


In [19]:
# Merging both sections on year

permits = total_rows.merge(residential_rows, on="year", how="inner")

# Converting types

permits["year"] = permits["year"].astype(int)
permits["building_permits_total"] = pd.to_numeric(permits["building_permits_total"], errors="coerce").astype(int)
permits["building_permits_residential"] = pd.to_numeric(permits["building_permits_residential"], errors="coerce").astype(int)

# Filtering to target window of 2005-2025

permits = permits[permits["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", permits.shape)
print("Year range:", permits["year"].min(), "to", permits["year"].max())
print("Missing values:\n", permits.isnull().sum())
permits.head()

Shape after cleaning: (21, 3)
Year range: 2005 to 2025
Missing values:
 year                            0
building_permits_total          0
building_permits_residential    0
dtype: int64


,year,building_permits_total,building_permits_residential
0,2005,242123,198896
1,2006,247986,201314
2,2007,188274,141405
3,2008,184059,133666
4,2009,184564,136078


In [20]:
# Saving cleaned building permits dataset

permits.to_csv("data_clean/building_permits_clean.csv", index=False)
print("Saved cleaned building permits dataset to data_clean/building_permits_clean.csv")

Saved cleaned building permits dataset to data_clean/building_permits_clean.csv


# House Price Index

**Residential Property Price Indices: Germany** - [(Source)](https://www-genesis.destatis.de/datenbank/online/statistic/61262/table/61262-0001/search/s/JTIyaG91c2UlMjBwcmljZSUyMGluZGV4JTIy)

- National house price indices covering overall, new, and existing residential properties
- Base year 2015=100
- Annual frequency, covers Germany as a whole
- Used as national-level price benchmark to validate city-level trends

In [24]:
# Dataset Transformation for House Price Index

hpi_raw = pd.read_excel("data_raw/61262-0001_en.xlsx", header=None)

# Inspecting Data Shape

print("HPI Shape:", hpi_raw.shape)

# Inspecting Columns

print("HPI Columns:", hpi_raw.columns.tolist())

# Inspecting Data Types

print("HPI Data Types:\n", hpi_raw.dtypes)

# Inspecting Missing Values

print("HPI Missing Values:\n", hpi_raw.isnull().sum())

# Dataset Preview

print("HPI Preview:\n", hpi_raw.head())

HPI Shape: (13, 52)
HPI Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
HPI Data Types:
 0         str
1         str
2     float64
3         str
4     float64
5         str
6     float64
7         str
8     float64
9         str
10    float64
11        str
12    float64
13        str
14    float64
15        str
16    float64
17        str
18    float64
19        str
20    float64
21        str
22    float64
23        str
24    float64
25        str
26    float64
27        str
28    float64
29        str
30    float64
31        str
32    float64
33        str
34    float64
35        str
36    float64
37        str
38    float64
39        str
40    float64
41        str
42    float64
43        str
44    float64
45        str
46     object
47        str
48     object
49        str
50     object
51        str
dtype: object
HPI Missi

/Users/aditya/Desktop/Housing Analysis/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [26]:
# Row 3 contains years spread across every other column starting from column 2, and each column has a flag column next to it

header_row = hpi_raw.iloc[3]

# Build year 

year_cols = {}
for i, val in enumerate(header_row):
    numeric_val = pd.to_numeric(val, errors="coerce")
    if pd.notna(numeric_val) and 1990 <= numeric_val <= 2030:
        year_cols[int(numeric_val)] = i

print("Years found:", sorted(year_cols.keys()))
print("Total years found:", len(year_cols))

# Extract values for each indicator using the year->col mapping

hpi = pd.DataFrame({
    "year": list(year_cols.keys()),
    "hpi_overall": [pd.to_numeric(hpi_raw.iloc[4, c], errors="coerce") for c in year_cols.values()],
    "hpi_new_property" : [pd.to_numeric(hpi_raw.iloc[5, c], errors="coerce") for c in year_cols.values()],
    "hpi_existing_property" : [pd.to_numeric(hpi_raw.iloc[6, c], errors="coerce") for c in year_cols.values()]
})

# Filtering to target window of 2005-2025

hpi = hpi[hpi["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", hpi.shape)
print("Year range:", hpi["year"].min(), "to", hpi["year"].max())
print("Missing values:\n", hpi.isnull().sum())
hpi.head()

Years found: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Total years found: 25
Shape after cleaning: (20, 4)
Year range: 2005 to 2024
Missing values:
 year                     0
hpi_overall              0
hpi_new_property         0
hpi_existing_property    0
dtype: int64


,year,hpi_overall,hpi_new_property,hpi_existing_property
0,2005,83.3,74.6,84.9
1,2006,83.0,73.9,84.6
2,2007,81.2,75.4,82.2
3,2008,82.3,77.4,83.2
4,2009,83.0,81.6,83.2


In [27]:
# Saving cleaned house price index dataset

hpi.to_csv("data_clean/house_price_index_clean.csv", index=False)
print("Saved cleaned house price index dataset to data_clean/house_price_index_clean.csv")

Saved cleaned house price index dataset to data_clean/house_price_index_clean.csv


# Headline CPI Dataset

**Consumer Price Index: Germany** - [(Source)](https://www-genesis.destatis.de/datenbank/online/statistic/61111/table/61111-0001/search/s/JTIyNjExMTElMjI%3D)

- Overall consumer price index for Germany, base year 2020=100
- Includes annual year-on-year percentage change
- Annual frequency
- Used as general inflation indicator and deflator for real price calculations

In [28]:
# Dataset Transformation for Headline CPI

cpi_raw = pd.read_excel("data_raw/61111-0001_en.xlsx", header=None)

# Inspecting Data Shape

print("CPI Shape:", cpi_raw.shape)

# Inspecting Columns

print("CPI Columns:", cpi_raw.columns.tolist())

# Inspecting Data Types

print("CPI Data Types:\n", cpi_raw.dtypes)

# Inspecting Missing Values

print("CPI Missing Values:\n", cpi_raw.isnull().sum())

# Dataset Preview

print("CPI Preview:\n", cpi_raw.head())

/Users/aditya/Desktop/Housing Analysis/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


CPI Shape: (43, 5)
CPI Columns: [0, 1, 2, 3, 4]
CPI Data Types:
 0       str
1    object
2       str
3    object
4       str
dtype: object
CPI Missing Values:
 0    2
1    6
2    8
3    6
4    9
dtype: int64
CPI Preview:
                                       0                     1    2  \
0  Consumer price index: Germany, years                   NaN  NaN   
1      Consumer price index for Germany                   NaN  NaN   
2                               Germany                   NaN  NaN   
3                                  Year  Consumer price index  NaN   
4                                   NaN              2020=100  NaN   

               3    4  
0            NaN  NaN  
1            NaN  NaN  
2            NaN  NaN  
3  Annual change  NaN  
4         in (%)  NaN  


In [29]:
# Data starts at row 5, columns (0 = year, 1 = CPI Index, 3 = YoY change), odd columns are flags

cpi = cpi_raw.iloc[5:, [0, 1, 3]].copy()
cpi.columns = ["year", "cpi_headline", "cpi_yoy_change_pct"]

# Convert to numeric

cpi["year"] = pd.to_numeric(cpi["year"], errors="coerce")
cpi["cpi_headline"] = pd.to_numeric(cpi["cpi_headline"], errors="coerce")
cpi["cpi_yoy_change_pct"] = pd.to_numeric(cpi["cpi_yoy_change_pct"], errors="coerce")

# Drop metadata rows at bottom

cpi = cpi.dropna(subset=["year", "cpi_headline"]).copy()
cpi["year"] = cpi["year"].astype(int)

# Filter to target window of 2005-2025

cpi = cpi[cpi["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", cpi.shape)
print("Year range:", cpi["year"].min(), "to", cpi["year"].max())
print("Missing values:\n", cpi.isnull().sum())
cpi.head()

Shape after cleaning: (21, 3)
Year range: 2005 to 2025
Missing values:
 year                  0
cpi_headline          0
cpi_yoy_change_pct    0
dtype: int64


,year,cpi_headline,cpi_yoy_change_pct
0,2005,81.5,1.6
1,2006,82.8,1.6
2,2007,84.7,2.3
3,2008,86.9,2.6
4,2009,87.2,0.3


In [30]:
# Saving cleaned headline cpi dataset

cpi.to_csv("data_clean/cpi_clean.csv", index=False)
print("Saved cleaned headline cpi dataset to data_clean/cpi_clean.csv")

Saved cleaned headline cpi dataset to data_clean/cpi_clean.csv


# CPI by Category - Housing Dataset

**Consumer Price Index by Category: Germany** - [(Source)](https://www-genesis.destatis.de/datenbank/online/statistic/61111/table/61111-0005/search/s/JTIyNjExMTElMjI%3D)

- CPI broken down by individual consumption purpose (COICOP classification)
- Base year 2020=100
- Extracting CC13-04: Housing, water, electricity, gas and other fuels
- Used as housing-specific inflation indicator, separates shelter costs from general inflation

In [31]:
# Data Transformation for CPI by category

cpi_cat_raw = pd.read_excel("data_raw/61111-0005_en.xlsx", header=None)

# Inspecting Data Shape

print("CPI by Category Shape:", cpi_cat_raw.shape)

# Inspecting Columns

print("CPI by Category Columns:", cpi_cat_raw.columns.tolist())

# Inspecting Data Types

print("CPI by Category Data Types:\n", cpi_cat_raw.dtypes)

# Inspecting Missing Values

print("CPI by Category Missing Values:\n", cpi_cat_raw.isnull().sum())

# Dataset Preview

print("CPI by Category Preview:\n", cpi_cat_raw.head())

CPI by Category Shape: (20, 72)
CPI by Category Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71]
CPI by Category Data Types:
 0         str
1         str
2     float64
3         str
4     float64
       ...   
67        str
68    float64
69        str
70    float64
71        str
Length: 72, dtype: object
CPI by Category Missing Values:
 0     1
1     8
2     7
3     8
4     7
     ..
67    8
68    7
69    8
70    7
71    8
Length: 72, dtype: int64
CPI by Category Preview:
                                                   0    1       2    3   \
0  Consumer price index: Germany, years, individu...  NaN     NaN  NaN   
1                   Consumer price index for Germany  NaN     NaN  NaN   
2                                            Germany  NaN    

/Users/aditya/Desktop/Housing Analysis/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [32]:
# Row 4 contains years as column headers

header_row = cpi_cat_raw.iloc[4]

# Build year->col mapping 

year_cols = {}
for i, val in enumerate(header_row):
    numeric_val = pd.to_numeric(val, errors="coerce")
    if pd.notna(numeric_val) and 1990 <= numeric_val <= 2030:
        year_cols[int(numeric_val)] = i

print("Years found:", sorted(year_cols.keys()))
print("Total years found:", len(year_cols))

# Find the housing category row (CC13-04)

housing_row = cpi_cat_raw[cpi_cat_raw[0].astype(str).str.contains("CC13-04", na=False)].index[0]

print("Housing category row index:", housing_row)
print("Row preview:\n", cpi_cat_raw.iloc[housing_row].tolist())

Years found: [1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Total years found: 35
Housing category row index: 8
Row preview:
 ['CC13-04', 'Housing, water, electricity, gas and other fuels', np.float64(52.0), 'e', np.float64(56.6), 'e', np.float64(61.2), 'e', np.float64(63.7), 'e', np.float64(65.6), 'e', np.float64(67.2), 'e', np.float64(68.9), 'e', np.float64(69.6), 'e', np.float64(70.5), 'e', np.float64(72.5), 'e', np.float64(74.2), 'e', np.float64(74.9), 'e', np.float64(76.0), 'e', np.float64(77.2), 'e', np.float64(79.3), 'e', np.float64(81.7), 'e', np.float64(83.3), 'e', np.float64(86.1), 'e', np.float64(86.5), 'e', np.float64(87.4), 'e', np.float64(90.0), 'e', np.float64(92.0), 'e', np.float64(93.9), 'e', np.float64(94.7), 'e', np.float64(94.3), 'e', np.float64(94.3), 'e', np.float64(95.5), 'e', np.float64(97.2), 'e', np

In [33]:
# Extract housing CPI values across all years

cpi_housing = pd.DataFrame({
    "year": list(year_cols.keys()), 
    "cpi_housing": [pd.to_numeric(cpi_cat_raw.iloc[housing_row, c], errors="coerce") for c in year_cols.values()]
})

# Filter to target window of 2005-2025

cpi_housing = cpi_housing[cpi_housing["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", cpi_housing.shape)
print("Year range:", cpi_housing["year"].min(), "to", cpi_housing["year"].max())
print("Missing values:\n", cpi_housing.isnull().sum())
cpi_housing.head()

Shape after cleaning: (21, 2)
Year range: 2005 to 2025
Missing values:
 year           0
cpi_housing    0
dtype: int64


,year,cpi_housing
0,2005,79.3
1,2006,81.7
2,2007,83.3
3,2008,86.1
4,2009,86.5


In [34]:
# Saving cleaned cpi by category dataset

cpi_housing.to_csv("data_clean/cpi_housing_clean.csv", index=False)
print("Saved cleaned cpi by category dataset to data_clean/cpi_housing_clean.csv")

Saved cleaned cpi by category dataset to data_clean/cpi_housing_clean.csv


# Wohngeld (Housing Allowance) Recipients

**Households Receiving Housing Allowance: Germany** - [(Source)](https://www-genesis.destatis.de/datenbank/online/statistic/22311/table/22311-0002/search/s/JTIyMjIzMTElMjI%3D)

- Annual count of households receiving state housing subsidies (Wohngeld)
- Broken down by rent support and mortgage/home upkeep support
- Measured on December 31 each yaer
- Used as housing affordability stress indicator at population level

In [35]:
# Dataset Transformation for Wohngeld Recipients

wohngeld_raw = pd.read_excel("data_raw/22311-0002_en.xlsx", header=None)

# Inspecting Data Shape

print("Wohngeld Shape:", wohngeld_raw.shape)

# Inspecting Columns

print("Wohngeld Columns:", wohngeld_raw.columns.tolist())

# Inspecting Data Types

print("Wohngeld Data Types:\n", wohngeld_raw.dtypes)

# Inspecting Missing Values

print("Wohngeld Missing Values:\n", wohngeld_raw.isnull().sum())

# Dataset Preview

print("Wohngeld Preview:\n", wohngeld_raw.head())

Wohngeld Shape: (29, 13)
Wohngeld Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Wohngeld Data Types:
 0        str
1     object
2        str
3     object
4        str
5     object
6        str
7     object
8        str
9     object
10       str
11    object
12       str
dtype: object
Wohngeld Missing Values:
 0     2
1     7
2     9
3     8
4     9
5     8
6     9
7     7
8     9
9     8
10    9
11    8
12    9
dtype: int64
Wohngeld Preview:
                                                   0   \
0  Households receiving housing allowance: German...   
1                   Housing allowance on 31 December   
2                                            Germany   
3    Households receiving housing allowance (number)   
4                                     Reference date   

                                                  1    2    3    4    5    6   \
0                                                NaN  NaN  NaN  NaN  NaN  NaN   
1                                               

/Users/aditya/Desktop/Housing Analysis/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [36]:
# Data starts from row 6, columns (0 = date, 1 = rent support, 3 = mortgage support, 5 = total)

wohngeld = wohngeld_raw.iloc[6:, [0, 1, 3, 5]].copy()
wohngeld.columns = ["date", "wohngeld_rent_support", "wohngeld_mortgage_support", "wohngeld_total"]

# Extract year from date string

wohngeld["year"] = pd.to_datetime(wohngeld["date"], errors="coerce").dt.year

# Drop date column and rows without valid year

wohngeld = wohngeld.dropna(subset=["year"]).copy()
wohngeld["year"] = wohngeld["year"].astype(int)

# Convert value columns to numeric

for col in ["wohngeld_rent_support", "wohngeld_mortgage_support", "wohngeld_total"]:
    wohngeld[col] = pd.to_numeric(wohngeld[col], errors="coerce")

# Drop date column and reorder

wohngeld = wohngeld.drop(columns=["date"])
wohngeld = wohngeld[["year", "wohngeld_rent_support", "wohngeld_mortgage_support", "wohngeld_total"]]

# Convert to integer

for col in ["wohngeld_rent_support", "wohngeld_mortgage_support", "wohngeld_total"]:
    wohngeld[col] = wohngeld[col].astype(int)

# Filter to target window of 2005-2025

wohngeld = wohngeld[wohngeld["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", wohngeld.shape)
print("Year range:", wohngeld["year"].min(), "to", wohngeld ["year"].max())
print("Missing values:\n", wohngeld.isnull().sum())
wohngeld.head()

Shape after cleaning: (20, 4)
Year range: 2005 to 2024
Missing values:
 year                         0
wohngeld_rent_support        0
wohngeld_mortgage_support    0
wohngeld_total               0
dtype: int64


,year,wohngeld_rent_support,wohngeld_mortgage_support,wohngeld_total
0,2005,695231,85429,780660
1,2006,591285,74607,665892
2,2007,517679,62623,580302
3,2008,522416,61619,584035
4,2009,775609,84001,859610


In [37]:
# Saving cleaned wohngeld dataset

wohngeld.to_csv("data_clean/wohngeld_clean.csv", index=False)
print("Saved cleaned wohngeld dataset to data_clean/wohngeld_clean.csv")

Saved cleaned wohngeld dataset to data_clean/wohngeld_clean.csv


# Earnings Index

**Indices of Collectively Agreed Earnings: Germany** - [(Source)](https://www-genesis.destatis.de/datenbank/online/statistic/62221/table/62221-0001/search/s/NjIyMjE%3D)

- Index of collectively agreed monthly earnings for the overall economy
- Base year 2020=100, covers both with and without special payments
- Annual frequency, overall economy (WZ08-A-3) section only
- Used as wage growth indicator for purchasing power analysis
- Note: Reliable data available from 2010 onwards

In [38]:
# Dataset Transformation for Earnings Index

earnings_raw = pd.read_excel("data_raw/62221-0001_en.xlsx", header=None)

# Inspecting Data Shape

print("Earnings Index Shape:", earnings_raw.shape)

# Inspecting Columns

print("Earnings Index Columns:", earnings_raw.columns.tolist())

# Inspecting Data Types

print("Earnings Index Data Types:\n", earnings_raw.dtypes)

# Inspecting Missing Values

print("Earnings Index Missing Values:\n", earnings_raw.isnull().sum())

# Dataset Preview



Earnings Index Shape: (4072, 13)
Earnings Index Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Earnings Index Data Types:
 0        str
1     object
2        str
3     object
4        str
5     object
6        str
7     object
8        str
9     object
10       str
11    object
12       str
dtype: object
Earnings Index Missing Values:
 0        2
1      133
2     1660
3      133
4     2045
5      133
6     1660
7      133
8     2045
9      133
10    2045
11     133
12    2045
dtype: int64


/Users/aditya/Desktop/Housing Analysis/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [40]:
# File has multiple sections stacked vertically, "Overall Economy" section (WZ08-A-03) starts at row 5 and runs until the next section header

# Find the Overall Economy section

overall_start = earnings_raw[earnings_raw[0].astype(str).str.contains("WZ08-A-03", na=False)].index[0]

# Find the next section header after it

next_section = earnings_raw.iloc[overall_start+1:][ earnings_raw.iloc[overall_start+1:][0].astype(str).str.contains("WZ08-", na=False)].index[0]

# Extract just the overall economy rows

earnings_section = earnings_raw.iloc[overall_start+1:next_section].copy()

# Keep rows where col 0 is a valid year

earnings_section = earnings_section[pd.to_numeric(earnings_section[0], errors="coerce").between(1990, 2030)].copy()

# Columns: 0=year, 1=hourly without bonus, 3=hourly with bonus, 5=monthly without bonus, 7=monthly with bonus

earnings = pd.DataFrame({
    "year": pd.to_numeric(earnings_section[0], errors="coerce").astype(int),
    "earnings_index_monthly_with_bonus": pd.to_numeric(earnings_section[7], errors="coerce"), 
    "earnings_index_monthly_without_bonus": pd.to_numeric(earnings_section[5], errors="coerce")
})

# Filter to target window of 2005-2025

earnings = earnings[earnings["year"].between(2005, 2025)].reset_index(drop=True)

print("Shape after cleaning:", earnings.shape)
print("Year range:", earnings["year"].min(), "to", earnings["year"].max())
print("Missing values:\n", earnings.isnull().sum())
earnings.head()

Shape after cleaning: (21, 3)
Year range: 2005 to 2025
Missing values:
 year                                    0
earnings_index_monthly_with_bonus       5
earnings_index_monthly_without_bonus    5
dtype: int64


,year,earnings_index_monthly_with_bonus,earnings_index_monthly_without_bonus
0,2005,NaN,NaN
1,2006,NaN,NaN
2,2007,NaN,NaN
3,2008,NaN,NaN
4,2009,NaN,NaN


In [41]:
# Saving cleaned earnings index dataset

earnings.to_csv("data_clean/earnings_index_clean.csv", index=False)
print("Saved cleaned earnings index dataset to data_clean/earnings_index_clean.csv")

Saved cleaned earnings index dataset to data_clean/earnings_index_clean.csv


# Merging All Datasets - Macro-Economin Master Dataset

Merging all 10 cleaned datasets into a single master file.
- Base: Unemployment
- Join key: year
- Join type: Left join on all datasets
- Wage proxy derived post-merge: 'avg_compensation_per_employee_eur'
- Expected final shape: (21, 22)
- Expected NaNs: earnings columns (2005-2009), employment/wage proxy (2005-2006)

In [43]:
# Loading all cleaned datasets

unemployment = pd.read_csv("data_clean/unemployment_clean.csv")
compensation = pd.read_csv("data_clean/compensation_clean.csv")
employment = pd.read_csv("data_clean/employment_clean.csv")
mortgage = pd.read_csv("data_clean/mortgage_clean.csv")
permits = pd.read_csv("data_clean/building_permits_clean.csv")
hpi = pd.read_csv("data_clean/house_price_index_clean.csv")
cpi = pd.read_csv("data_clean/cpi_clean.csv")
cpi_housing = pd.read_csv("data_clean/cpi_housing_clean.csv")
wohngeld = pd.read_csv("data_clean/wohngeld_clean.csv")
earnings = pd.read_csv("data_clean/earnings_index_clean.csv")

print("All datasets loaded successfully.")
print("Shapes:")
for name, df in [("unemployment", unemployment),
                 ("compensation", compensation),
                 ("employment", employment), 
                 ("mortgage", mortgage),
                 ("permits", permits),
                 ("hpi", hpi),
                 ("cpi", cpi),
                 ("cpi_housing", cpi_housing),
                 ("wohngeld", wohngeld),
                 ("earnings",earnings)]:
    print(f"{name}: {df.shape}")

All datasets loaded successfully.
Shapes:
unemployment: (21, 2)
compensation: (21, 2)
employment: (19, 2)
mortgage: (21, 4)
permits: (21, 3)
hpi: (20, 4)
cpi: (21, 3)
cpi_housing: (21, 2)
wohngeld: (20, 4)
earnings: (21, 3)


In [44]:
# Merging all datasets using left join on year

macro = unemployment.copy()

for name, df in [("compensation", compensation),
                 ("employment", employment), 
                 ("mortgage", mortgage),
                 ("permits", permits),
                 ("hpi", hpi),
                 ("cpi", cpi),
                 ("cpi_housing", cpi_housing),
                 ("wohngeld", wohngeld),
                 ("earnings",earnings)]:
    macro = macro.merge(df, on="year", how="left")
    print(f"After merging {name}: {macro.shape}")

After merging compensation: (21, 3)
After merging employment: (21, 4)
After merging mortgage: (21, 7)
After merging permits: (21, 9)
After merging hpi: (21, 12)
After merging cpi: (21, 14)
After merging cpi_housing: (21, 15)
After merging wohngeld: (21, 18)
After merging earnings: (21, 20)


In [45]:
# Deriving wage proxy - avg compensation per employee (compensation_bn_eur / persons_in_employment_mill * 1000)

macro["avg_compensation_per_employee_eur"] = ((macro["compensation_bn_eur"] / macro["persons_in_employment_mill"]) * 1000).round(0)

print("Wage Proxy derived successfully.")
print("Sample values:")
print(macro[["year", "compensation_bn_eur", "persons_in_employment_mill", "avg_compensation_per_employee_eur"]].dropna().head())

Wage Proxy derived successfully.
Sample values:
   year  compensation_bn_eur  persons_in_employment_mill  \
2  2007             1224.813                      37.153   
3  2008             1273.169                      37.637   
4  2009             1281.143                      37.887   
5  2010             1320.515                      37.412   
6  2011             1377.957                      38.178   

   avg_compensation_per_employee_eur  
2                            32967.0  
3                            33828.0  
4                            33815.0  
5                            35297.0  
6                            36093.0  


In [46]:
# Reordering columns into logical groups

macro = macro[["year", # Time
               "unemployment_rate_pct", "persons_in_employment_mill", # Labour Market
               "compensation_bn_eur", "avg_compensation_per_employee_eur", "earnings_index_monthly_with_bonus", "earnings_index_monthly_without_bonus", # Wages
               "cpi_headline", "cpi_yoy_change_pct", "cpi_housing", # Inflation
               "building_permits_total", "building_permits_residential", # Housing Supply
               "hpi_overall", "hpi_new_property", "hpi_existing_property", # House Prices
               "mortgage_rate_avg_pct", "mortgage_rate_min_pct", "mortgage_rate_max_pct", # Credit Conditions
               "wohngeld_total", "wohngeld_rent_support", "wohngeld_mortgage_support"]] # Affordability Stress

print("Final shape:", macro.shape)
print("Final columns:", macro.columns.tolist())

Final shape: (21, 21)
Final columns: ['year', 'unemployment_rate_pct', 'persons_in_employment_mill', 'compensation_bn_eur', 'avg_compensation_per_employee_eur', 'earnings_index_monthly_with_bonus', 'earnings_index_monthly_without_bonus', 'cpi_headline', 'cpi_yoy_change_pct', 'cpi_housing', 'building_permits_total', 'building_permits_residential', 'hpi_overall', 'hpi_new_property', 'hpi_existing_property', 'mortgage_rate_avg_pct', 'mortgage_rate_min_pct', 'mortgage_rate_max_pct', 'wohngeld_total', 'wohngeld_rent_support', 'wohngeld_mortgage_support']


In [47]:
# Validation Checks

print("Macro-Economic Master Dataset Validation Checks")

print(f"\nRows: {len(macro)} (expected 21)")
print(f"Columns: {len(macro.columns)} (expected 21)")
print(f"\nYear Range: {macro['year'].min()} to {macro['year'].max()} (expected 2005 to 2025)")

print("\nNull counts per column:")
nulls = macro.isnull().sum()
for col, n in nulls.items():
    status = "OK" if n == 0 else "MISSING VALUES"
    print(f"{status} {col}: {n}")

print("\nSpot Checks")
print(f" Unemployment 2005: {macro.loc[macro['year']==2005, 'unemployment_rate_pct'].values[0]}% (expected 9.9)")
print(f" Mortgage Avg 2022: {macro.loc[macro['year']==2022, 'mortgage_rate_avg_pct'].values[0]}% (expected 2.57)")
print(f" HPI overall 2021: {macro.loc[macro['year']==2021, 'hpi_overall'].values[0]} (expected 154.7)")
print(f" CPI YoY 2022: {macro.loc[macro['year']==2022, 'cpi_yoy_change_pct'].values[0]}% (expected 6.9)")
print(f" Permits residential 2023: {macro.loc[macro['year']==2023, 'building_permits_residential'].values[0]} (expected 124167)")

Macro-Economic Master Dataset Validation Checks

Rows: 21 (expected 21)
Columns: 21 (expected 21)

Year Range: 2005 to 2025 (expected 2005 to 2025)

Null counts per column:
OK year: 0
OK unemployment_rate_pct: 0
MISSING VALUES persons_in_employment_mill: 2
OK compensation_bn_eur: 0
MISSING VALUES avg_compensation_per_employee_eur: 2
MISSING VALUES earnings_index_monthly_with_bonus: 5
MISSING VALUES earnings_index_monthly_without_bonus: 5
OK cpi_headline: 0
OK cpi_yoy_change_pct: 0
OK cpi_housing: 0
OK building_permits_total: 0
OK building_permits_residential: 0
MISSING VALUES hpi_overall: 1
MISSING VALUES hpi_new_property: 1
MISSING VALUES hpi_existing_property: 1
OK mortgage_rate_avg_pct: 0
OK mortgage_rate_min_pct: 0
OK mortgage_rate_max_pct: 0
MISSING VALUES wohngeld_total: 1
MISSING VALUES wohngeld_rent_support: 1
MISSING VALUES wohngeld_mortgage_support: 1

Spot Checks
 Unemployment 2005: 9.9% (expected 9.9)
 Mortgage Avg 2022: 2.57% (expected 2.57)
 HPI overall 2021: 154.7 (expec

In [48]:
# Final preview

macro.head()

,year,unemployment_rate_pct,persons_in_employment_mill,compensation_bn_eur,avg_compensation_per_employee_eur,earnings_index_monthly_with_bonus,earnings_index_monthly_without_bonus,cpi_headline,cpi_yoy_change_pct,cpi_housing,...,building_permits_residential,hpi_overall,hpi_new_property,hpi_existing_property,mortgage_rate_avg_pct,mortgage_rate_min_pct,mortgage_rate_max_pct,wohngeld_total,wohngeld_rent_support,wohngeld_mortgage_support
0,2005,9.9,NaN,1166.822,NaN,NaN,NaN,81.5,1.6,79.3,...,198896,83.3,74.6,84.9,4.34,4.18,4.55,780660.0,695231.0,85429.0
1,2006,9.1,NaN,1189.871,NaN,NaN,NaN,82.8,1.6,81.7,...,201314,83.0,73.9,84.6,4.69,4.40,4.87,665892.0,591285.0,74607.0
2,2007,7.4,37.153,1224.813,32967.0,NaN,NaN,84.7,2.3,83.3,...,141405,81.2,75.4,82.2,5.15,4.85,5.37,580302.0,517679.0,62623.0
3,2008,6.4,37.637,1273.169,33828.0,NaN,NaN,86.9,2.6,86.1,...,133666,82.3,77.4,83.2,5.27,4.96,5.54,584035.0,522416.0,61619.0
4,2009,6.9,37.887,1281.143,33815.0,NaN,NaN,87.2,0.3,86.5,...,136078,83.0,81.6,83.2,4.34,4.13,4.83,859610.0,775609.0,84001.0


In [49]:
# Saving final macro-economic master dataset

macro.to_csv("data_clean/macro_master_dataset.csv", index=False)
print("Saved final macro-economic master dataset to data_clean/macro_master_dataset.csv")

Saved final macro-economic master dataset to data_clean/macro_master_dataset.csv
